In [ ]:
from pathlib import Path

import polars as pl

from mtx.run_config import RunConfig


# ============================================================
# 1. Use the same parameters as the pipeline
# ============================================================

CONFIG = RunConfig(
    week_start="2026-02-02",
    week_end="2026-02-08",
    correlation_start="2026-02-02",
    correlation_end="2026-02-08",
    meter_correlation_radius=550,
    correlation_method="spearman",
    weekly_partition_size=25_000,
    seed_batch_size=5_000,
    force_ami_refresh=False,
)

PAIR_DIRECTORY = CONFIG.pair_directory

print(
    f"Checking folder:\n{PAIR_DIRECTORY}\n"
)


# ============================================================
# 2. Find all pair partitions
# ============================================================

pair_files = sorted(
    file_path
    for file_path in PAIR_DIRECTORY.rglob("part_*.parquet")
    if ".temporary." not in file_path.name
    and ".tmp." not in file_path.name
    and not any(
        "backup" in part.lower()
        for part in file_path.parts
    )
)

if not pair_files:
    raise FileNotFoundError(
        f"No pair partitions found under:\n{PAIR_DIRECTORY}"
    )

print(
    f"Pair partitions found: {len(pair_files):,}"
)


# ============================================================
# 3. Lazily read all partitions
# ============================================================

pair_frames = []

for file_path in pair_files:
    relative_path = file_path.relative_to(
        PAIR_DIRECTORY
    )

    frame = (
        pl.scan_parquet(file_path)
        .with_columns(
            pl.lit(
                str(relative_path)
            ).alias("_source_file")
        )
    )

    pair_frames.append(frame)

all_pairs = pl.concat(
    pair_frames,
    how="vertical",
)


# ============================================================
# 4. Create a canonical pair
#
# A-B and B-A are treated as the same pair.
# ============================================================

canonical_pairs = all_pairs.with_columns(
    pl.when(
        pl.col("meter_1")
        <= pl.col("meter_2")
    )
    .then(pl.col("meter_1"))
    .otherwise(pl.col("meter_2"))
    .alias("_pair_first"),

    pl.when(
        pl.col("meter_1")
        <= pl.col("meter_2")
    )
    .then(pl.col("meter_2"))
    .otherwise(pl.col("meter_1"))
    .alias("_pair_second"),

    pl.concat_str(
        [
            pl.col("meter_1"),
            pl.lit(" -> "),
            pl.col("meter_2"),
        ]
    ).alias("_direction"),
)


# ============================================================
# 5. Find duplicate or reversed pairs
# ============================================================

duplicate_pairs = (
    canonical_pairs
    .group_by(
        [
            "division",
            "_pair_first",
            "_pair_second",
        ]
    )
    .agg(
        pl.len().alias("occurrences"),

        pl.col("_direction")
        .n_unique()
        .alias("direction_count"),

        pl.col("_direction")
        .unique()
        .alias("directions"),

        pl.col("_source_file")
        .unique()
        .alias("source_files"),

        pl.col("distance_ft")
        .min()
        .alias("minimum_distance_ft"),

        pl.col("distance_ft")
        .max()
        .alias("maximum_distance_ft"),

        pl.col("correlation")
        .min()
        .alias("minimum_correlation"),

        pl.col("correlation")
        .max()
        .alias("maximum_correlation"),
    )
    .filter(
        pl.col("occurrences") > 1
    )
    .with_columns(
        pl.when(
            pl.col("direction_count") > 1
        )
        .then(pl.lit("reversed pair"))
        .otherwise(pl.lit("exact duplicate"))
        .alias("duplicate_type")
    )
    .sort(
        [
            "division",
            "_pair_first",
            "_pair_second",
        ]
    )
    .collect()
)


# ============================================================
# 6. Display results
# ============================================================

if duplicate_pairs.is_empty():
    print(
        "\nPASS: No duplicate or reversed pairs found."
    )

else:
    duplicate_summary = (
        duplicate_pairs
        .group_by(
            [
                "division",
                "duplicate_type",
            ]
        )
        .agg(
            pl.len().alias(
                "duplicate_pair_groups"
            ),

            (
                pl.col("occurrences") - 1
            )
            .sum()
            .alias("extra_rows"),
        )
        .sort(
            [
                "division",
                "duplicate_type",
            ]
        )
    )

    print(
        f"\nDuplicate pair groups found: "
        f"{duplicate_pairs.height:,}"
    )

    print(
        f"Extra duplicate rows: "
        f"{int((duplicate_pairs['occurrences'] - 1).sum()):,}"
    )

    print("\nSummary by division:")

    display(duplicate_summary)

    print("\nFirst 50 duplicate pairs:")

    display(
        duplicate_pairs.select(
            [
                "division",
                "_pair_first",
                "_pair_second",
                "duplicate_type",
                "occurrences",
                "directions",
                "source_files",
                "minimum_distance_ft",
                "maximum_distance_ft",
                "minimum_correlation",
                "maximum_correlation",
            ]
        ).head(50)
    )


# ============================================================
# 7. Save the diagnostic results
# ============================================================

diagnostic_path = (
    PAIR_DIRECTORY
    / "duplicate_pair_diagnostics.parquet"
)

duplicate_pairs.write_parquet(
    diagnostic_path
)

print(
    f"\nDiagnostic results saved to:\n"
    f"{diagnostic_path}"
)

Checking folder:
C:\Users\u355394\OneDrive - IBERDROLA S.A\Documents - Asset Data Strategy\Asset Data Quality\SmartLink\data\processed\meter_pairs\week_20260202_20260208\correlation_20260202_20260208\radius_550ft\spearman\seed_batch_5000

Pair partitions found: 249
